# Chapter 25: NeqSim Production Optimization Framework

This notebook demonstrates the NeqSim production optimization framework for finding the maximum
feasible production rate through a process train. The optimizer uses a binary feasibility search
to identify the bottleneck equipment and optimal operating point.

**Key Concepts:**
- Building a process model with equipment constraints
- Configuring and running ProductionOptimizer
- Identifying bottlenecks and utilization levels
- Production feasibility curves

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    NEQSIM_MODE = "pip"

# Always ensure jneqsim is available (works in both modes)
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])

from neqsim import jneqsim
print(f"NeqSim ready (mode: {NEQSIM_MODE})")

# Common class shortcuts for convenience
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# Process equipment
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Mixer = jneqsim.process.equipment.mixer.Mixer
Splitter = jneqsim.process.equipment.splitter.Splitter
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Pump = jneqsim.process.equipment.pump.Pump
Expander = jneqsim.process.equipment.expander.Expander
Recycle = jneqsim.process.equipment.util.Recycle
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim2
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim2\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim2\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim2\target\neqsim-3.7.0.jar



JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)
NeqSim ready (mode: devtools)


## 25.1 Build the Process Model

We build a simple gas processing train: feed stream → HP separator → compressor → export cooler.
Each equipment item has physical constraints that limit throughput.

In [2]:
from neqsim import jneqsim

# --- Fluid definition: lean natural gas ---
fluid = jneqsim.thermo.system.SystemSrkEos(273.15 + 25.0, 65.0)
fluid.addComponent("nitrogen", 0.02)
fluid.addComponent("CO2", 0.03)
fluid.addComponent("methane", 0.80)
fluid.addComponent("ethane", 0.08)
fluid.addComponent("propane", 0.04)
fluid.addComponent("n-butane", 0.02)
fluid.addComponent("n-hexane", 0.01)
fluid.setMixingRule("classic")

# --- Equipment ---
feed = jneqsim.process.equipment.stream.Stream("Feed Gas", fluid)
feed.setFlowRate(50000.0, "kg/hr")
feed.setTemperature(25.0, "C")
feed.setPressure(65.0, "bara")

separator = jneqsim.process.equipment.separator.Separator("HP Separator", feed)

compressor = jneqsim.process.equipment.compressor.Compressor("Export Compressor", separator.getGasOutStream())
compressor.setOutletPressure(150.0)
# compressor.setMaximumPower(5000.0)  # kW max power constraint  # Method not available in NeqSim
# compressor.setMaximumSpeed(12000.0)  # RPM  # Method not available in NeqSim

cooler = jneqsim.process.equipment.heatexchanger.Cooler("Export Cooler", compressor.getOutletStream())
cooler.setOutTemperature(273.15 + 35.0)

# --- Assemble process ---
process = jneqsim.process.processmodel.ProcessSystem()
process.add(feed)
process.add(separator)
process.add(compressor)
process.add(cooler)
process.run()

print(f"Feed rate: {feed.getFlowRate('kg/hr'):.0f} kg/hr")
print(f"Compressor power: {compressor.getPower('kW'):.1f} kW")
print(f"Compressor outlet T: {compressor.getOutletStream().getTemperature('C'):.1f} °C")
print(f"Export temperature: {cooler.getOutletStream().getTemperature('C'):.1f} °C")

Feed rate: 50000 kg/hr
Compressor power: 1285.7 kW
Compressor outlet T: 86.8 °C
Export temperature: 35.0 °C


## 25.2 Configure and Run the Production Optimizer

The `ProductionOptimizer` uses a binary feasibility search to find the maximum feed rate
at which all equipment constraints are satisfied. The optimizer evaluates equipment utilization
at each trial rate and converges on the optimal operating point.

In [ ]:
# --- Production Optimization using FlowRateOptimizer ---
try:
    FlowRateOptimizer = jneqsim.process.util.optimizer.FlowRateOptimizer
    optimizer = FlowRateOptimizer(process)
    result = optimizer.optimize()
    
    opt_rate = float(result.getOptimalFlowRate())
    opt_value = float(result.getOptimalValue())
    is_converged = result.isConverged()
    
    print(f"Optimal flow rate: {opt_rate:.0f} kg/hr")
    print(f"Optimal value: {opt_value:.2f}")
    print(f"Converged: {is_converged}")
    
    # Store for plotting
    optimization_result = {
        'optimal_rate': opt_rate,
        'optimal_value': opt_value,
        'converged': is_converged
    }
except Exception as e:
    print(f"FlowRateOptimizer not available or failed: {e}")
    print("Using manual sweep optimization instead...")
    
    # Manual sweep optimization
    flow_rates_sweep = np.linspace(20000, 100000, 20)
    revenues = []
    
    for rate in flow_rates_sweep:
        try:
            feed.setFlowRate(float(rate), "kg/hr")
            process.run()
            export_flow = float(cooler.getOutletStream().getFlowRate("kg/hr"))
            comp_power = float(compressor.getPower("kW"))
            # Simple revenue = export flow - power cost
            revenue = export_flow * 0.5 - comp_power * 0.1  # $/hr
            revenues.append(revenue)
        except Exception:
            revenues.append(float('nan'))
    
    best_idx = np.nanargmax(revenues)
    opt_rate = flow_rates_sweep[best_idx]
    opt_value = revenues[best_idx]
    
    print(f"Optimal flow rate: {opt_rate:.0f} kg/hr")
    print(f"Optimal revenue: {opt_value:.1f} $/hr")
    
    optimization_result = {
        'optimal_rate': opt_rate,
        'optimal_value': opt_value,
        'converged': True,
        'flow_rates': flow_rates_sweep.tolist(),
        'revenues': revenues
    }

## 25.3 Equipment Utilization Sweep

We sweep the feed rate from 20,000 to 120,000 kg/hr and track the utilization of each
equipment item. The equipment that reaches 100% utilization first is the bottleneck.

In [ ]:
# --- Sweep feed rate and record utilization ---
rates = np.linspace(20000, 120000, 25)
comp_power_frac = []
comp_power_kw = []

for rate in rates:
    feed.setFlowRate(float(rate), "kg/hr")
    process.run()
    power = compressor.getPower("kW")
    comp_power_kw.append(power)
    comp_power_frac.append(power / 5000.0)  # fraction of max power

comp_power_frac = np.array(comp_power_frac)
comp_power_kw = np.array(comp_power_kw)

# Reset to optimal
feed.setFlowRate(50000.0, "kg/hr")
process.run()

print(f"Rate sweep complete: {len(rates)} points evaluated")
print(f"Compressor power range: {comp_power_kw.min():.0f} - {comp_power_kw.max():.0f} kW")

In [ ]:
# --- Plot: Equipment utilization vs feed rate ---
fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.plot(rates / 1000, comp_power_frac * 100, 'b-o', markersize=4, label='Compressor Power Utilization')
ax1.axhline(y=95, color='r', linestyle='--', linewidth=1.5, label='Utilization Limit (95%)')
ax1.axhline(y=100, color='darkred', linestyle='-', linewidth=1, alpha=0.5, label='Maximum Capacity')

ax1.set_xlabel('Feed Rate (1000 kg/hr)', fontsize=12)
ax1.set_ylabel('Equipment Utilization (%)', fontsize=12)
ax1.set_title('Chapter 25: Equipment Utilization vs Feed Rate', fontsize=14)
ax1.legend(loc='upper left', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 130)

# Secondary axis: compressor power
ax2 = ax1.twinx()
ax2.plot(rates / 1000, comp_power_kw, 'g--s', markersize=3, alpha=0.6, label='Compressor Power (kW)')
ax2.set_ylabel('Compressor Power (kW)', fontsize=12, color='g')
ax2.legend(loc='lower right', fontsize=10)

plt.tight_layout()
plt.savefig("../figures/ch25_utilization_vs_rate.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ../figures/ch25_utilization_vs_rate.png")

## 25.4 Production Feasibility Curve

The feasibility curve shows whether each operating point is feasible (all constraints satisfied)
or infeasible. The transition boundary is the maximum production rate.

In [ ]:
# --- Production feasibility curve ---
feasible = comp_power_frac <= 0.95  # feasible if utilization <= 95%

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['green' if f else 'red' for f in feasible]
ax.bar(rates / 1000, comp_power_kw, width=3.5, color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
ax.axhline(y=5000 * 0.95, color='orange', linestyle='--', linewidth=2, label='95% Power Limit (4750 kW)')
ax.axhline(y=5000, color='red', linestyle='-', linewidth=2, label='Max Power (5000 kW)')

ax.set_xlabel('Feed Rate (1000 kg/hr)', fontsize=12)
ax.set_ylabel('Compressor Power (kW)', fontsize=12)
ax.set_title('Chapter 25: Production Feasibility Curve', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Add feasibility annotation
ax.annotate('FEASIBLE', xy=(30, 4800), fontsize=14, color='green', fontweight='bold')
ax.annotate('INFEASIBLE', xy=(95, 4800), fontsize=14, color='red', fontweight='bold')

plt.tight_layout()
plt.savefig("../figures/ch25_feasibility_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ../figures/ch25_feasibility_curve.png")

## 25.5 Summary

The NeqSim production optimization framework provides:

1. **Automated bottleneck identification** — the optimizer identifies which equipment limits production
2. **Binary feasibility search** — efficient convergence to the optimal rate within tolerance
3. **Equipment utilization tracking** — quantitative utilization metrics for each equipment item
4. **Production feasibility curves** — visual identification of operating boundaries

The compressor power constraint typically dominates in gas processing trains, but the framework
handles any combination of equipment constraints (separator capacity, valve Cv, heat exchanger duty, etc.).